# 📄 PDFPlumber 实战教程

> 配套文档：《[PDFPLUMBER前沿项目巡礼.md](PDFPLUMBER前沿项目巡礼.md)》（本目录）
> 示例文件：`background-checks.pdf`（FBI NICS 枪支背景检查月度报表，来自 [官方仓库 examples/pdfs](https://github.com/jsvine/pdfplumber/tree/stable/examples/pdfs)）
> 调研对象：https://github.com/jsvine/pdfplumber（v0.11.x，MIT 协议）

## 本节课你会学会

| 章节 | 内容 | 关键词 |
|:---|:---|:---|
| 1️⃣ 打开 PDF | `pdfplumber.open()`、`PDF` 类 | `metadata` / `pages` |
| 2️⃣ 对象模型 | `chars` / `lines` / `rects` / … | `x0` / `top` / `bottom` |
| 3️⃣ 文本提取 | `extract_text()` | `x_tolerance` / `layout` |
| 4️⃣ 词级提取 | `extract_words()` | `extra_attrs` / `return_chars` |
| 5️⃣ 搜索 | `search()` | 正则 / `case=False` |
| 6️⃣ 裁剪与过滤 | `crop` / `within_bbox` / `filter` | bbox |
| 7️⃣ 表格提取 | `extract_table()` / `find_tables()` | `table_settings` |
| 8️⃣ 视觉调试 | `to_image()` / `debug_tablefinder()` | 叠加可视化 |
| 9️⃣ 其他技巧 | `dedupe_chars` / 多页遍历 / CLI | 内存管理 |

> ⚠️ 本文所有代码单元格均可在本 notebook 中**按顺序直接运行**，输出为对 `background-checks.pdf` 的真实解析结果。

---

## 🛠️ 环境准备

需要 Python 3.10+。运行下面的单元格安装 `pdfplumber`（本机已安装 0.11.9，若已安装会提示 Requirement already satisfied）。

In [ ]:
%pip install pdfplumber -q

---

## 1️⃣ 打开 PDF — `pdfplumber.open()`

`pdfplumber.open(x)` 中的 `x` 可以是：PDF 文件**路径**、以字节加载的**文件对象**、或**类文件对象**（如 `io.BytesIO`）。

它返回 `pdfplumber.PDF` 类的实例，核心有两个属性：

- `.metadata`：PDF 元数据字典（来自 Info trailer，通常含 Producer / CreationDate / ModDate）
- `.pages`：每页一个 `pdfplumber.Page` 实例的列表

推荐用 `with` 语句，结束时自动关闭文件并释放资源。

In [ ]:
import pdfplumber

with pdfplumber.open("background-checks.pdf") as pdf:
    print("PDF 类名:", type(pdf).__name__)
    print("PDF 页数:", len(pdf.pages))
    print("元数据:", pdf.metadata)

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]                      # 第 1 页
    print("页面编号:", page.page_number)
    print("页面宽度:", page.width, "pt | 高度:", page.height, "pt")
    print("mediabox:", page.mediabox)
    print("cropbox:", page.cropbox)

---

## 2️⃣ 对象模型：每个字符/线/矩形都是 dict

pdfplumber 把页面拆成基础对象，**每个对象是一个普通 Python dict**：

| 属性 | 内容 |
|:---|:---|
| `.chars` | 每个文本字符一个对象 |
| `.lines` | 一维线段 |
| `.rects` | 二维矩形 |
| `.curves` | pdfminer 不认为是线/矩形的连续点路径 |
| `.images` / `.annots` / `.hyperlinks` | 图像 / 注释 / 超链接 |

**坐标体系**（很重要）：`x0/x1` 距**左边缘**、`top/bottom` 距**顶部**、`y0/y1` 距**底部**、`doctop` 距**文档顶部**（跨页连续）。

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    print("字符 chars      :", len(page.chars))
    print("线条 lines      :", len(page.lines))
    print("矩形 rects      :", len(page.rects))
    print("曲线 curves     :", len(page.curves))
    print("图像 images     :", len(page.images))
    print("注释 annots     :", len(page.annots))
    print("超链接 hyperlinks:", len(page.hyperlinks))

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    c = page.chars[0]                       # 页面第 1 个字符
    print("字符内容:", repr(c["text"]))
    print("字体:", c["fontname"], "| 字号:", c["size"])
    print("位置 x0/x1:", round(c["x0"], 2), "/", round(c["x1"], 2))
    print("位置 top/bottom:", round(c["top"], 2), "/", round(c["bottom"], 2))
    print("宽/高:", round(c["width"], 2), "/", round(c["height"], 2))
    print("全部字段:", sorted(c.keys()))

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    ln = page.lines[0]
    print("第 1 条线:", {k: round(ln[k], 1) for k in ("x0", "x1", "top", "bottom")}, "| 线宽:", ln["linewidth"])
    rt = page.rects[0]
    print("第 1 个矩形:", {k: round(rt[k], 1) for k in ("x0", "x1", "top", "bottom")})
    print("矩形描边色:", rt["stroking_color"], "| 填充色:", rt["non_stroking_color"])
    print("派生边 rect_edges:", len(page.rect_edges))
    print("全部边 edges:", len(page.edges))

---

## 3️⃣ 文本提取 — `extract_text()`

`extract_text()` 把页面所有字符对象拼成字符串，规则如下：

- **空格**：当上一字符 `x1` 与下一字符 `x0` 的间距 > `x_tolerance`（默认 3）时补一个空格
- **换行**：当上一字符与下一字符的 `doctop` 差 > `y_tolerance`（默认 3）时换行
- `layout=True`（实验性）：按 `x_density=7.25` / `y_density=13` 尽量**还原页面版式**（表格列会垂直对齐）

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    text = page.extract_text()
    print("--- 前 300 字符 ---")
    print(text[:300])

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    t1 = page.extract_text(x_tolerance=1)      # 字间距 > 1pt 就算空格（更细）
    t10 = page.extract_text(x_tolerance=10)    # 字间距 > 10pt 才算空格（更粗）
    print("两种容差结果是否相同:", t1 == t10)
    for a, b in zip(t1.splitlines(), t10.splitlines()):
        if a != b:
            print("x_tolerance=1 :", a[:90])
            print("x_tolerance=10:", b[:90])
            break

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    layout = page.extract_text(layout=True)      # 保留版式（实验性）
    lines = [ln.rstrip() for ln in layout.splitlines() if ln.strip()]
    print("layout=True 的非空行数:", len(lines))
    print("\n".join(lines[:12]))

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    text_lines = page.extract_text_lines()       # 实验性 API：逐行返回文本+坐标
    print("检测到文本行数:", len(text_lines))
    print("每行的字段:", sorted(text_lines[0].keys()))
    for tl in text_lines[:3]:
        print("行文本:", tl["text"][:50], "| x0:", round(tl["x0"], 1), "| top:", round(tl["top"], 1))

---

## 4️⃣ 词级提取 — `extract_words()`

把相邻字符聚合成**词**，每个词返回一个带包围盒坐标的 dict。常用参数：

- `extra_attrs=["fontname", "size"]`：同词内的字符必须共享这些属性完全相同的值
- `return_chars=True`：词 dict 里附带组成它的字符列表（`"chars"` 字段）
- `split_at_punctuation=True`：在标点处强制断词（默认连字如 `ﬁ` 会展开成 `fi`）
- `use_text_flow=True`：按 PDF 底层字符流顺序而非坐标排序（类似鼠标划选）

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    words = page.extract_words()
    print("词总数:", len(words))
    for w in words[:3]:
        print(w["text"], {k: round(w[k], 1) for k in ("x0", "x1", "top", "bottom")})

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    words = page.extract_words(extra_attrs=["size"], return_chars=True)
    print("词数:", len(words))
    print("词的字段:", sorted(words[0].keys()))
    print("首词:", words[0]["text"], "| size:", words[0]["size"], "| 组成字符:", len(words[0]["chars"]))
    print("默认词数:", len(page.extract_words()), "| 按标点切分:", len(page.extract_words(split_at_punctuation=True)))

---

## 5️⃣ 搜索 — `search()`（实验性）

在页面上搜索文本，返回**所有匹配实例**：每个实例包含匹配文本、正则分组、包围盒坐标和字符对象。参数：

- `pattern`：编译/未编译的正则，或普通字符串（`regex=False`）
- `case=False`：忽略大小写
- `main_group`：只返回特定正则分组
- 零宽与纯空白匹配会被丢弃（它们没有明确位置）

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    hits = page.search("Alabama")
    print("精确匹配 'Alabama':", len(hits), "处")
    h = hits[0]
    print("首处坐标:", (round(h["x0"], 1), round(h["top"], 1), round(h["x1"], 1), round(h["bottom"], 1)))
    print("忽略大小写 'nics':", len(page.search("nics", case=False)), "处")
    nums = page.search(r"\d{2}")
    print("正则 \\d{2} (2位数字):", len(nums), "处, 首个:", nums[0]["text"])

---

## 6️⃣ 裁剪与过滤 — `crop()` / `within_bbox()` / `filter()`

bbox 格式为 4 元组 `(x0, top, x1, bottom)`。三个方法都返回**新页面对象**（不修改原页面）：

| 方法 | 保留的对象 |
|:---|:---|
| `.crop(bbox)` | 与 bbox **部分或全部重叠**的对象（跨界的会被切到框内） |
| `.within_bbox(bbox)` | **完全在** bbox 内的对象 |
| `.outside_bbox(bbox)` | **完全在** bbox 外的对象 |
| `.filter(fn)` | `fn(obj)` 返回 `True` 的对象 |

> 常用套路：**先裁剪出目标区域，再 extract_text / extract_table**。`relative=True` 表示 bbox 相对页面左上角偏移；`strict=False` 允许 bbox 越出页面。

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    col = page.crop((40, 45, 300, page.height))   # 州名列所在的窄区域
    print("裁剪后区域大小:", round(col.width, 1), "x", round(col.height, 1))
    words = col.extract_words()
    print("区域内的词数:", len(words))
    print("前 8 个词:", [w["text"] for w in words[:8]])

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    top_area = page.within_bbox((0, 0, page.width, 200))      # 完全在顶部 200pt 内
    rest = page.outside_bbox((0, 0, page.width, 200))         # 完全在框外
    print("完全位于顶部区域(0,0,1008,200)内的字符:", len(top_area.chars))
    print("完全位于区域外的字符:", len(rest.chars))
    print("两类合计:", len(top_area.chars) + len(rest.chars), "(全部字符", len(page.chars), ")")
    big = page.filter(lambda obj: obj["object_type"] != "char" or obj["size"] > 10)
    print("过滤后保留的字符(字号>10):", len(big.chars))

---

## 7️⃣ 表格提取 — `extract_table()`

表格检测算法（来自 Anssi Nurminen 硕士论文 + Tabula 启发）五步：**找线（含文字对齐暗示的“虚边”）→ 合并近似线 → 求交点 → 以交点为顶点生成最小单元格 → 聚合为表格**。

| API | 返回 |
|:---|:---|
| `.extract_table()` | 页面上**最大**表格，`行 → 单元格` 的两层列表 |
| `.extract_tables()` | 页面上**所有**表格，`表 → 行 → 单元格` 三层列表 |
| `.find_table()` / `.find_tables()` | `Table` 对象（`.cells/.rows/.columns/.bbox` + `.extract()`） |
| `.debug_tablefinder()` | `TableFinder` 对象（`.edges/.intersections/.cells/.tables`） |

`table_settings` 里 `vertical_strategy` / `horizontal_strategy` 可取值：`"lines"`（图形线）、`"lines_strict"`（不含矩形边）、`"text"`（文字对齐虚边）、`"explicit"`（手动指定线）。

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    table = page.extract_table()
    print("表格:", len(table), "行 x", len(table[0]), "列")
    print("行[0] (标题):", table[0][0])
    print("行[1] (列名) 前3格:", table[1][:3])
    print("行[2] (数据) 前3格:")
    for cell in table[2][:3]:
        print("   ", repr(cell))
    print("行[-1] (合计) 前3格:", table[-1][:3])

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    settings = {
        "vertical_strategy": "lines",
        "horizontal_strategy": "lines",
        "snap_tolerance": 3,          # 平行线对齐容差
        "edge_min_length": 3,         # v0.11.8+ 还有 edge_min_length_prefilter 可捕获虚线
    }
    rows = page.extract_table(settings)
    print("自定义设置提取:", len(rows), "行 x", len(rows[0]), "列")
    tables = page.find_tables()
    print("find_tables 检测到:", len(tables), "个表格")
    t = tables[0]
    print("Table: 行", len(t.rows), "| 列", len(t.columns), "| 单元格", len(t.cells))
    print("表格 bbox:", [round(v, 1) for v in t.bbox])
    print("第3行第1列:", t.extract()[2][0][:40])

---

## 8️⃣ 视觉调试 — `to_image()` 与 `debug_tablefinder()`

把页面渲染成图像（基于 pypdfium2），再把检测结果叠加画上去，肉眼验证解析是否正确：

- `.to_image(resolution=100)`：按 100 像素/英寸渲染；还支持 `width` / `height` / `antialias` / `force_mediabox`
- `im.debug_tablefinder(...)`：叠加检测结果——**线=红色、交点=圆圈、表格=浅蓝**
- `im.draw_rects / draw_lines / draw_circles / draw_vline / draw_hline ...`：手工绘制（SVG 风格 `fill`/`stroke`/`stroke_width` 参数），可传对象或坐标
- `im.save("xxx.png")`：保存；Jupyter 中直接显示为单元格输出

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    im = page.to_image(resolution=100)          # 100 像素/英寸
    print("图像尺寸:", im.original.size)
    im.debug_tablefinder(table_settings={})     # 叠加: 线=红, 交点=圆圈, 表格=浅蓝
    im.save("debug-tablefinder.png")
    print("已保存 debug-tablefinder.png（检测线=红、交点=圆圈、表格=浅蓝）")

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    im = page.to_image(resolution=100)
    big = [c for c in page.chars if c["size"] > 10]   # 大字号字符（标题等）
    im.draw_rects(big, stroke=(255, 0, 0, 255), stroke_width=2)      # 红色框
    im.draw_circles(big[:5], radius=4, fill=(0, 255, 0, 255))        # 绿色圆点标前 5 个
    im.save("demo-draw.png")
    print("已保存 demo-draw.png | 框出的字符:", len(big))

---

## 9️⃣ 其他实用功能

1. **`.dedupe_chars()`**：删除“相同文本 + 相同位置（容差内）+ 相同属性”的重复字符（某些 PDF 有叠加字符层）
2. **多页遍历**：`pdf.pages` 逐个处理即可（本示例文件只有 1 页）
3. **`open()` 其他参数**：

   | 参数 | 作用 |
   |:---|:---|
   | `password="..."` | 密码保护的 PDF |
   | `laparams={"line_overlap": 0.7}` | 传给 pdfminer 布局分析引擎的参数 |
   | `unicode_norm="NFKC"` | Unicode 规范化（v0.11.3+，处理复合字符/全角符号） |
   | `strict_metadata=True` | 元数据解析失败直接抛异常（默认仅警告） |
   | `repair=True` | 先用 Ghostscript 修复损坏 PDF 再打开 |

4. **内存管理**：`Page` 会缓存解析结果；解析大型 PDF 时调 `page.close()`（或 `pdf.close()`）冲刷缓存释放内存。

In [ ]:
with pdfplumber.open("background-checks.pdf") as pdf:
    page = pdf.pages[0]
    deduped = page.dedupe_chars()
    print("去重前后字符数:", len(page.chars), "->", len(deduped.chars))

# 多页 PDF 的遍历写法（本示例只有 1 页，语法通用）
with pdfplumber.open("background-checks.pdf") as pdf:
    for page in pdf.pages:
        first_line = page.extract_text().splitlines()[0]
        print(f"第 {page.page_number} 页 | {len(page.chars)} chars | 首行: {first_line!r}")

---

## 🖥️ 命令行 CLI（不用写代码也能用）

安装后自带 `pdfplumber` 命令。在**本目录的终端**里执行：

```bash
# 把 PDF 中每个字符/线/矩形导出为 CSV（官方 README 示例）
pdfplumber background-checks.pdf > background-checks.csv

# JSON 格式（含页面级元数据与嵌套属性）
pdfplumber background-checks.pdf --format json

# 纯文本（等价于 Page.extract_text(layout=True)）
pdfplumber background-checks.pdf --format text

# 只提取第 1 页的 char 对象，坐标保留 2 位小数
pdfplumber background-checks.pdf --pages "1" --types char --precision 2

# 传入 pdfminer 布局分析参数
pdfplumber background-checks.pdf --laparams '{"detect_vertical": true}'
```

---

## 🏋️ 综合练习

在下方新单元格中完成（提示：先写 `with pdfplumber.open("background-checks.pdf") as pdf:` 再取 `page = pdf.pages[0]`）：

1. **统计出现次数**：用 `page.search("nics", case=False)` 统计本页出现 "NICS" 的次数，并打印全部命中位置。

2. **提取全部州名**：`extract_table()` 结果的第 2 行（`table[2]`）第 1 列包含每行若干个州名（以 `\n` 分隔）。把它按 `\n` 拆开，统计**总共有多少个州**，并与 `page.search` 数到的州名数量对比。

3. **数据清洗**：把 `table[2]` 之后的每一行第 1 列拆分，做出 `{州名: 总数}` 的字典（总数在第 25 列）。打印前 3 个州的总额。
   ```python
   # 参考骨架
   with pdfplumber.open("background-checks.pdf") as pdf:
       table = pdf.pages[0].extract_table()
       states, totals = [], []
       for row in table[2:]:
           names = (row[0] or "").split("\n")
           total = row[-1]
           ...  # 请补全
   ```

4. **进阶**：用 `find_tables()[0].bbox` 裁剪页面并重新提取表格，验证结果是否一致（体会 "先裁剪再提取" 的套路）。

5. **挑战**：把 `extract_table()` 的结果保存为 CSV 文件（可用 `csv` 标准库），再用 Excel/记事本打开检查。

---

## 🔗 参考链接

- 本课程配套文档：[PDFPLUMBER前沿项目巡礼.md](PDFPLUMBER前沿项目巡礼.md)（系统性的项目技术巡礼）
- 仓库主页：https://github.com/jsvine/pdfplumber
- README（API 完整文档）：https://github.com/jsvine/pdfplumber/blob/stable/README.md
- CHANGELOG（版本演进）：https://github.com/jsvine/pdfplumber/blob/stable/CHANGELOG.md
- 示例 PDF 与 notebook：https://github.com/jsvine/pdfplumber/tree/stable/examples
- 文档目录（颜色/修复/结构树）：https://github.com/jsvine/pdfplumber/tree/stable/docs
- 讨论区（表格提取疑难杂症汇集地）：https://github.com/jsvine/pdfplumber/discussions
- 底层解析引擎 pdfminer.six：https://github.com/pdfminer/pdfminer.six